# 03 Feature Stats + ML
It contributes:

- Feature engineering: epochs the continuous trials and labels `MC`, `MD`, `MF`.
- Hypothesis-driven metrics: extracts frontal beta, parietal/occipital alpha, and computes Focus Index (beta/alpha).
- Alpha stability metric: quantifies alpha fluctuation (std/var) across epochs per trial.
- Group statistics: runs repeated-measures ANOVA and paired tests (`MD` vs `MC`, `MD` vs `MF`) to test condition effects.
- Baseline ML: runs leave-one-subject-out classification (`MD` vs `MC`) to check if interference condition is decodable across people.
- Reproducible outputs: saves clean CSV tables in outputs/ for figures, reporting, and your final presentation.


In [1]:
from __future__ import annotations

import argparse
import glob
import os
import re
from dataclasses import dataclass
from typing import Dict, List, Sequence, Tuple

import numpy as np
import pandas as pd
import scipy.io
from scipy.signal import welch
from scipy.stats import rankdata, ttest_rel
from statsmodels.stats.anova import AnovaRM


MUSIC_CONDITIONS = ("MC", "MD", "MF")
FRONTAL_CHANNELS = ("F3", "F4", "FZ")
PARIETO_OCCIPITAL_CHANNELS = ("P3", "P4", "PZ", "O1", "O2")


@dataclass
class DatasetInfo:
    subject: str
    fs: float
    trials: List[dict]


In [2]:
def parse_condition(label: str) -> str:
    m = re.search(r"_(MC|MD|MF|SD|SF)\.$", str(label))
    return m.group(1) if m else "UNK"


def parse_song_id(label: str) -> int | None:
    m = re.match(r"^(\d+)_", str(label))
    return int(m.group(1)) if m else None


def parse_subject_from_path(path: str) -> str:
    name = os.path.basename(path)
    m = re.match(r"(S\d+)_preprocessed\.mat$", name)
    return m.group(1) if m else os.path.splitext(name)[0]


def flatten_channel_name(obj: object) -> str:
    if isinstance(obj, np.ndarray):
        if obj.size == 0:
            return ""
        return flatten_channel_name(obj.flat[0])
    return str(obj).strip()


def load_channel_names(path: str) -> List[str]:
    m = scipy.io.loadmat(path, simplify_cells=True)
    raw = m.get("ch_list")
    if raw is None:
        raise ValueError("channel_list.mat missing 'ch_list'")

    arr = np.asarray(raw).ravel()
    labels = [flatten_channel_name(x).upper() for x in arr]
    if not labels:
        raise ValueError("No channels found in channel_list.mat")
    return labels


def get_channel_indices(all_channels: Sequence[str], wanted: Sequence[str]) -> Tuple[List[int], List[str]]:
    idx_map = {ch.upper(): i for i, ch in enumerate(all_channels)}
    found_idx: List[int] = []
    missing: List[str] = []
    for ch in wanted:
        key = ch.upper()
        if key in idx_map:
            found_idx.append(idx_map[key])
        else:
            missing.append(ch)
    return found_idx, missing


def load_subject_data(path: str) -> DatasetInfo:
    m = scipy.io.loadmat(path, simplify_cells=True)
    data = m["Data"]
    info = data["info"]
    fs = float(info["EEG_fs"])
    return DatasetInfo(subject=parse_subject_from_path(path), fs=fs, trials=data["data"])


def make_epoch_starts(n_samples: int, epoch_samples: int, step_samples: int) -> np.ndarray:
    if n_samples < epoch_samples:
        return np.array([], dtype=int)
    return np.arange(0, n_samples - epoch_samples + 1, step_samples, dtype=int)


def bandpower_welch(
    epoch_data: np.ndarray,
    fs: float,
    low_hz: float,
    high_hz: float,
    nperseg: int,
) -> float:
    """
    epoch_data shape: (n_channels, n_samples)
    Returns mean PSD power in [low_hz, high_hz] averaged over channels.
    """
    freqs, psd = welch(epoch_data, fs=fs, axis=-1, nperseg=nperseg)
    mask = (freqs >= low_hz) & (freqs <= high_hz)
    if not np.any(mask):
        return float("nan")
    return float(np.mean(psd[:, mask]))


def compute_fold_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_score: np.ndarray) -> Dict[str, float]:
    y_true = y_true.astype(int)
    y_pred = y_pred.astype(int)
    n = y_true.size

    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))

    accuracy = (tp + tn) / n if n else np.nan
    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    balanced_accuracy = np.nanmean([sensitivity, specificity])
    precision = tp / (tp + fp) if (tp + fp) else np.nan
    recall = sensitivity
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else np.nan

    # AUC from ranks; equivalent to Mann-Whitney U based formulation.
    pos = y_true == 1
    neg = y_true == 0
    if np.any(pos) and np.any(neg):
        ranks = rankdata(y_score)
        auc = (np.sum(ranks[pos]) - np.sum(np.arange(1, np.sum(pos) + 1))) / (np.sum(pos) * np.sum(neg))
    else:
        auc = np.nan

    return {
        "n_test": n,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": float(auc),
    }


def paired_test(df_wide: pd.DataFrame, cond_a: str, cond_b: str) -> Dict[str, float]:
    pair = df_wide[[cond_a, cond_b]].dropna()
    a = pair[cond_a].to_numpy()
    b = pair[cond_b].to_numpy()
    if len(pair) < 2:
        return {
            "contrast": f"{cond_a}_vs_{cond_b}",
            "n_subjects": len(pair),
            "mean_diff": np.nan,
            "t_stat": np.nan,
            "p_value": np.nan,
            "cohen_d_paired": np.nan,
        }
    t_stat, p_value = ttest_rel(a, b)
    diff = a - b
    d = np.mean(diff) / np.std(diff, ddof=1) if np.std(diff, ddof=1) > 0 else np.nan
    return {
        "contrast": f"{cond_a}_vs_{cond_b}",
        "n_subjects": len(pair),
        "mean_diff": float(np.mean(diff)),
        "t_stat": float(t_stat),
        "p_value": float(p_value),
        "cohen_d_paired": float(d),
    }


def run_loso_md_vs_mc(epoch_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, str]:
    """
    Binary classification on epochs for MD vs MC with LOSO validation.
    Uses sklearn SVM if available; otherwise falls back to statsmodels logistic regression.
    """
    bin_df = epoch_df[epoch_df["condition"].isin(["MD", "MC"])].copy()
    if bin_df.empty:
        return pd.DataFrame(), pd.DataFrame(), "none"

    X = bin_df[["beta_frontal", "alpha_parietal", "focus_index"]].to_numpy(float)
    y = (bin_df["condition"] == "MD").astype(int).to_numpy()
    groups = bin_df["subject"].to_numpy()
    subjects = np.unique(groups)

    fold_rows: List[dict] = []
    y_true_all: List[np.ndarray] = []
    y_pred_all: List[np.ndarray] = []
    y_score_all: List[np.ndarray] = []

    try:
        from sklearn.pipeline import make_pipeline  # type: ignore
        from sklearn.preprocessing import StandardScaler  # type: ignore
        from sklearn.svm import SVC  # type: ignore

        model_name = "sklearn_linear_svm"
        for test_subj in subjects:
            test_mask = groups == test_subj
            train_mask = ~test_mask
            X_train, y_train = X[train_mask], y[train_mask]
            X_test, y_test = X[test_mask], y[test_mask]

            clf = make_pipeline(StandardScaler(), SVC(kernel="linear", probability=True, class_weight="balanced"))
            clf.fit(X_train, y_train)
            y_score = clf.predict_proba(X_test)[:, 1]
            y_pred = (y_score >= 0.5).astype(int)

            metrics = compute_fold_metrics(y_test, y_pred, y_score)
            metrics["test_subject"] = test_subj
            fold_rows.append(metrics)
            y_true_all.append(y_test)
            y_pred_all.append(y_pred)
            y_score_all.append(y_score)
    except Exception:
        import statsmodels.api as sm

        model_name = "statsmodels_logit_fallback"
        for test_subj in subjects:
            test_mask = groups == test_subj
            train_mask = ~test_mask
            X_train, y_train = X[train_mask], y[train_mask]
            X_test, y_test = X[test_mask], y[test_mask]

            mu = X_train.mean(axis=0)
            sigma = X_train.std(axis=0)
            sigma[sigma == 0] = 1.0
            X_train_z = (X_train - mu) / sigma
            X_test_z = (X_test - mu) / sigma

            X_train_c = sm.add_constant(X_train_z, has_constant="add")
            X_test_c = sm.add_constant(X_test_z, has_constant="add")

            try:
                model = sm.Logit(y_train, X_train_c).fit(disp=False)
            except Exception:
                model = sm.Logit(y_train, X_train_c).fit_regularized(alpha=1e-4, disp=False)

            y_score = np.asarray(model.predict(X_test_c), dtype=float)
            y_pred = (y_score >= 0.5).astype(int)

            metrics = compute_fold_metrics(y_test, y_pred, y_score)
            metrics["test_subject"] = test_subj
            fold_rows.append(metrics)
            y_true_all.append(y_test)
            y_pred_all.append(y_pred)
            y_score_all.append(y_score)

    fold_df = pd.DataFrame(fold_rows).sort_values("test_subject").reset_index(drop=True)
    y_true_cat = np.concatenate(y_true_all) if y_true_all else np.array([], dtype=int)
    y_pred_cat = np.concatenate(y_pred_all) if y_pred_all else np.array([], dtype=int)
    y_score_cat = np.concatenate(y_score_all) if y_score_all else np.array([], dtype=float)
    overall = compute_fold_metrics(y_true_cat, y_pred_cat, y_score_cat) if y_true_cat.size else {}
    overall["model"] = model_name
    overall_df = pd.DataFrame([overall])
    return fold_df, overall_df, model_name


In [3]:
def run_pipeline(args: argparse.Namespace) -> None:
    os.makedirs(args.out_dir, exist_ok=True)
    mat_files = sorted(glob.glob(os.path.join(args.data_dir, "S*_preprocessed.mat")))
    if not mat_files:
        raise FileNotFoundError(f"No subject files found in: {args.data_dir}")

    ch_names = load_channel_names(args.channel_list)
    frontal_idx, missing_frontal = get_channel_indices(ch_names, FRONTAL_CHANNELS)
    parocc_idx, missing_parocc = get_channel_indices(ch_names, PARIETO_OCCIPITAL_CHANNELS)

    if missing_frontal or missing_parocc:
        raise ValueError(
            "Missing required channels: "
            f"frontal_missing={missing_frontal}, parocc_missing={missing_parocc}"
        )

    epoch_rows: List[dict] = []
    trial_rows: List[dict] = []
    fs_values: List[float] = []

    for path in mat_files:
        ds = load_subject_data(path)
        fs_values.append(ds.fs)

        epoch_samples = int(round(args.epoch_sec * ds.fs))
        step_samples = int(round(epoch_samples * (1.0 - args.overlap)))
        if epoch_samples <= 0 or step_samples <= 0:
            raise ValueError("Invalid epoching parameters; check --epoch-sec and --overlap")
        nperseg = min(epoch_samples, int(round(args.nperseg_sec * ds.fs)))
        if nperseg < 8:
            raise ValueError("Welch nperseg is too small; increase --nperseg-sec or --epoch-sec")

        for trial_idx, trial in enumerate(ds.trials):
            cond = parse_condition(trial["TargetType"])
            if cond not in MUSIC_CONDITIONS:
                continue

            eeg = np.asarray(trial["EEG"], dtype=float)
            if eeg.ndim != 2:
                continue
            if eeg.shape[0] != len(ch_names) and eeg.shape[1] == len(ch_names):
                eeg = eeg.T
            if eeg.shape[0] != len(ch_names):
                continue

            n_samples = eeg.shape[1]
            starts = make_epoch_starts(n_samples, epoch_samples, step_samples)
            alpha_vals: List[float] = []

            for epoch_idx, start in enumerate(starts):
                stop = start + epoch_samples
                epoch = eeg[:, start:stop]
                frontal = epoch[frontal_idx, :]
                parocc = epoch[parocc_idx, :]

                beta_frontal = bandpower_welch(frontal, ds.fs, 13.0, 30.0, nperseg)
                alpha_par = bandpower_welch(parocc, ds.fs, 8.0, 12.0, nperseg)
                focus_index = beta_frontal / alpha_par if alpha_par > 0 else np.nan
                alpha_vals.append(alpha_par)

                epoch_rows.append(
                    {
                        "subject": ds.subject,
                        "file": os.path.basename(path),
                        "trial_index": trial_idx,
                        "trial_label": str(trial["TargetType"]),
                        "song_id": parse_song_id(trial["TargetType"]),
                        "condition": cond,
                        "epoch_index": epoch_idx,
                        "start_sample": int(start),
                        "end_sample": int(stop),
                        "beta_frontal": beta_frontal,
                        "alpha_parietal": alpha_par,
                        "focus_index": focus_index,
                    }
                )

            alpha_arr = np.asarray(alpha_vals, dtype=float)
            trial_rows.append(
                {
                    "subject": ds.subject,
                    "file": os.path.basename(path),
                    "trial_index": trial_idx,
                    "trial_label": str(trial["TargetType"]),
                    "song_id": parse_song_id(trial["TargetType"]),
                    "condition": cond,
                    "n_epochs": int(alpha_arr.size),
                    "alpha_mean": float(np.nanmean(alpha_arr)) if alpha_arr.size else np.nan,
                    "alpha_std": float(np.nanstd(alpha_arr, ddof=1)) if alpha_arr.size > 1 else np.nan,
                    "alpha_var": float(np.nanvar(alpha_arr, ddof=1)) if alpha_arr.size > 1 else np.nan,
                }
            )

    epoch_df = pd.DataFrame(epoch_rows)
    trial_df = pd.DataFrame(trial_rows)
    if epoch_df.empty:
        raise RuntimeError("No epoch features extracted. Check dataset paths/labels.")

    # Clean non-finite values produced by edge cases.
    for col in ["beta_frontal", "alpha_parietal", "focus_index"]:
        epoch_df[col] = pd.to_numeric(epoch_df[col], errors="coerce")
    epoch_df = epoch_df.dropna(subset=["beta_frontal", "alpha_parietal", "focus_index"])
    trial_df = trial_df.dropna(subset=["alpha_std", "alpha_var"], how="all")

    # Subject-level aggregation for repeated-measures statistics.
    subj_cond_focus = (
        epoch_df.groupby(["subject", "condition"], as_index=False)["focus_index"]
        .mean()
        .sort_values(["subject", "condition"])
    )
    subj_cond_alpha_stability = (
        trial_df.groupby(["subject", "condition"], as_index=False)[["alpha_std", "alpha_var"]]
        .mean()
        .sort_values(["subject", "condition"])
    )

    # rmANOVA: Focus Index
    focus_complete = subj_cond_focus.copy()
    cond_n = focus_complete.groupby("subject")["condition"].nunique()
    keep_subjects = cond_n[cond_n == len(MUSIC_CONDITIONS)].index
    focus_complete = focus_complete[focus_complete["subject"].isin(keep_subjects)]
    aov_focus = AnovaRM(focus_complete, depvar="focus_index", subject="subject", within=["condition"]).fit()
    aov_focus_tbl = aov_focus.anova_table.reset_index().rename(columns={"index": "effect"})

    focus_wide = focus_complete.pivot(index="subject", columns="condition", values="focus_index")
    focus_posthoc = pd.DataFrame(
        [
            paired_test(focus_wide, "MD", "MC"),
            paired_test(focus_wide, "MD", "MF"),
        ]
    )
    focus_posthoc["p_bonferroni"] = np.minimum(focus_posthoc["p_value"] * 2, 1.0)

    # rmANOVA: Alpha stability (std)
    alpha_complete = subj_cond_alpha_stability.copy()
    cond_n_alpha = alpha_complete.groupby("subject")["condition"].nunique()
    keep_subjects_alpha = cond_n_alpha[cond_n_alpha == len(MUSIC_CONDITIONS)].index
    alpha_complete = alpha_complete[alpha_complete["subject"].isin(keep_subjects_alpha)]
    aov_alpha_std = AnovaRM(alpha_complete, depvar="alpha_std", subject="subject", within=["condition"]).fit()
    aov_alpha_std_tbl = aov_alpha_std.anova_table.reset_index().rename(columns={"index": "effect"})

    alpha_wide = alpha_complete.pivot(index="subject", columns="condition", values="alpha_std")
    alpha_posthoc = pd.DataFrame(
        [
            paired_test(alpha_wide, "MD", "MC"),
            paired_test(alpha_wide, "MD", "MF"),
        ]
    )
    alpha_posthoc["p_bonferroni"] = np.minimum(alpha_posthoc["p_value"] * 2, 1.0)

    # LOSO baseline ML.
    loso_folds, loso_overall, model_used = run_loso_md_vs_mc(epoch_df)

    # Save outputs.
    epoch_df.to_csv(os.path.join(args.out_dir, "epoch_features.csv"), index=False)
    trial_df.to_csv(os.path.join(args.out_dir, "trial_alpha_stability.csv"), index=False)
    subj_cond_focus.to_csv(os.path.join(args.out_dir, "subject_condition_focus.csv"), index=False)
    subj_cond_alpha_stability.to_csv(os.path.join(args.out_dir, "subject_condition_alpha_stability.csv"), index=False)
    aov_focus_tbl.to_csv(os.path.join(args.out_dir, "rmanova_focus_index.csv"), index=False)
    focus_posthoc.to_csv(os.path.join(args.out_dir, "posthoc_focus_index.csv"), index=False)
    aov_alpha_std_tbl.to_csv(os.path.join(args.out_dir, "rmanova_alpha_std.csv"), index=False)
    alpha_posthoc.to_csv(os.path.join(args.out_dir, "posthoc_alpha_std.csv"), index=False)
    loso_folds.to_csv(os.path.join(args.out_dir, "loso_md_vs_mc_folds.csv"), index=False)
    loso_overall.to_csv(os.path.join(args.out_dir, "loso_md_vs_mc_overall.csv"), index=False)

    print("=== Pipeline Complete ===")
    print(f"Subjects: {len(np.unique(epoch_df['subject']))}")
    print(f"Unique sampling rates: {sorted(set(fs_values))}")
    print(f"Epoch rows (MC/MD/MF): {len(epoch_df)}")
    print(f"Trial rows (MC/MD/MF): {len(trial_df)}")
    print("")
    print("Focus Index rmANOVA:")
    print(aov_focus_tbl.to_string(index=False))
    print("")
    print("Focus Index post-hoc:")
    print(focus_posthoc.to_string(index=False))
    print("")
    print("Alpha stability (std) rmANOVA:")
    print(aov_alpha_std_tbl.to_string(index=False))
    print("")
    print("Alpha stability post-hoc:")
    print(alpha_posthoc.to_string(index=False))
    print("")
    print(f"LOSO model: {model_used}")
    print(loso_overall.to_string(index=False))
    print("")
    print(f"Saved outputs to: {os.path.abspath(args.out_dir)}")


In [5]:
# Parameters
args = argparse.Namespace(
    data_dir='Preprocessed',
    channel_list='channel_list.mat',
    out_dir='outputs',
    epoch_sec=2.0,
    overlap=0.5,
    nperseg_sec=2.0,
)

run_pipeline(args)


=== Pipeline Complete ===
Subjects: 18
Unique sampling rates: [512.0]
Epoch rows (MC/MD/MF): 17940
Trial rows (MC/MD/MF): 260

Focus Index rmANOVA:
   effect  F Value  Num DF  Den DF   Pr > F
condition 1.410353     2.0    34.0 0.257974

Focus Index post-hoc:
contrast  n_subjects  mean_diff    t_stat  p_value  cohen_d_paired  p_bonferroni
MD_vs_MC          18   0.008536  0.920139 0.370373        0.216879      0.740746
MD_vs_MF          18  -0.006125 -0.974952 0.343254       -0.229798      0.686508

Alpha stability (std) rmANOVA:
   effect  F Value  Num DF  Den DF   Pr > F
condition 1.623591     2.0    34.0 0.212106

Alpha stability post-hoc:
contrast  n_subjects  mean_diff   t_stat  p_value  cohen_d_paired  p_bonferroni
MD_vs_MC          18   0.215376 1.327828 0.201790        0.312972      0.403579
MD_vs_MF          18   0.185598 1.584488 0.131507        0.373467      0.263015

LOSO model: statsmodels_logit_fallback
 n_test   tp   tn   fp   fn  accuracy  balanced_accuracy  precision   r